# Chapter 11 — Beyond The Cannon: a latent-variable model (Lux)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/the-cannon-tutorial/blob/main/notebooks/11_lux.ipynb)

**Goal.** Write *Lux* (Horta et al. 2025, H25) — a data-driven model in which each star has a hidden ("latent")
vector from which *both* its labels and its spectrum are generated — and compare it with The Cannon on the twelve
labels of Chapter 10. Find out what treating the labels as noisy outputs, rather than exact inputs, does and does
not buy.

**You will learn**
1. how a latent-variable model differs from The Cannon, and why its training is a sequence of weighted
   least-squares fits;
2. that a seemingly harmless choice — how to solve a least-squares problem that has more unknowns than data
   constrain — decides how many latent dimensions the model really uses;
3. to check a new method against simple baselines (ridge regression) and against the "measured or predicted?" test
   of Chapter 8 before crediting it with a gain;
4. where Lux is clearly better than The Cannon: when the training labels are noisy.

**The model** (H25 §3). Star $n$ has a latent vector $\mathbf z_n$ of length $P$. Its $M$ labels and its spectrum are

$$\boldsymbol\ell_n = A\,\mathbf z_n + \text{noise}, \qquad \mathbf f_n = B\,\mathbf z_n + \text{noise},$$

with $A$ an $M\times P$ and $B$ a $\Lambda\times P$ matrix ($\Lambda$ pixels). The label noise has the catalogue
errors; the flux noise has the flux errors plus a fitted scatter $s_\lambda$ per pixel, as in The Cannon. In The
Cannon the labels are exact inputs and the spectrum is a quadratic function of them; in Lux both are outputs, linear
in $\mathbf z$, and the labels' errors enter the fit. To label a new star, find its $\mathbf z$ from the spectrum
alone (a weighted least-squares fit, with no local minima), then $\boldsymbol\ell = A\mathbf z$.

**Training** (H25 §4.1–4.2). Start from $\mathbf z_n = [1, \text{scaled labels}, 0, \dots, 0]$ and random $A$, $B$.
Stage 1, five rounds without scatter: fit $A$ to the labels (with $\mathbf z$ fixed), $B$ to the spectra, then each
$\mathbf z_n$ to its star's labels and spectrum. Stage 2, once: fit the scatter, then $B$ and $\mathbf z$ again with
it (and a weak Gaussian prior of width $\tau = 1000$ on $\mathbf z$). The fits of $A$, $B$ and $\mathbf z$ are
linear in the parameters they fit and are solved exactly by weighted least squares; the scatter is fitted by maximum
likelihood on a grid, as in Chapter 2 — the block `lux.py` below, about sixty lines. For the same least-squares steps
our code and the independent *pollux* package agree to $10^{-9}$ of the errors (Appendix B).

One detail of H25's published code (its last step used the previous $B$ and scatter) differed from the paper; the
author fixed it on 2026-11-01 after we reported it. We follow the paper.

**Scope of these examples.** This chapter uses one split and one missing/noisy-label draw to introduce the
method. Appendix C repeats those experiments with training-only continuum selection, three random draws,
paired bootstrap intervals, per-label bias/RMSE tables, retuned ridge and underestimated Lux errors.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are extracts of APOGEE DR17 and Gaia DR3, published on Zenodo (https://doi.org/10.5281/zenodo.23813051).
# MANIFEST.json lists, for each file, the query that made it and its SHA-256 checksum.
DATA_URL = 'https://zenodo.org/records/23813051/files/{}?download=1'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL.format('MANIFEST.json')).read())


def load(name, cache=Path('data')):
    """Download <name> (.parquet table or .npz arrays) from Zenodo if needed, check its SHA-256, read it.

    A .parquet file is returned as a pandas DataFrame, an .npz file as a dict of numpy arrays.
    """
    ext = '.npz' if name.startswith(('spectra_', 'rvs_')) else '.parquet'
    path = cache / f'{name}{ext}'
    sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
    if not path.exists() or sha(path) != MANIFEST[name]['sha256']:   # first use, or a stale/damaged copy
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        part = path.with_suffix('.part')
        urllib.request.urlretrieve(DATA_URL.format(path.name), part)   # download to a temporary file first
        if sha(part) != MANIFEST[name]['sha256']:
            raise IOError(f'{path.name}: checksum does not match the manifest (incomplete download?)')
        part.replace(path)                                         # only a verified file takes the final name
    # the checksum guarantees that everyone runs this chapter on exactly the same data
    if ext == '.parquet':
        return pd.read_parquet(path)
    with np.load(path, allow_pickle=False) as f:   # allow_pickle=False: arrays only, no code can be loaded
        return {k: f[k] for k in f.files}
# --- Spectra and continuum normalisation (as in Chapters 0 and 3) ----------------------------------------
NPIX = 8575
wave = 10**(4.179 + 6e-6 * np.arange(NPIX))               # apStar vacuum wavelength grid [Angstrom]
CHIPS = [(15150, 15800), (15865, 16425), (16480, 16950)]  # the three detectors [A], inside the edges of Chapter 0
BAD_BITS = sum(1 << b for b in [0, 1, 2, 3, 4, 5, 6, 7, 12])   # pixels given zero weight (Chapter 0)


def prepare(sp, rows=slice(None)):
    """flux and err (float64) of the spectra file sp; err = inf where there are no data, a BAD_BITS flag, or
    outside CHIPS (the detector edges, where only some spectra have data)."""
    flux = sp['flux'][rows].astype(float)
    err = sp['err'][rows].astype(float)
    on_chip = np.any([(wave > lo) & (wave < hi) for lo, hi in CHIPS], axis=0)   # pixels at the detector edges and
    bad = ((sp['mask'][rows] & BAD_BITS) > 0) | ~np.isfinite(flux) | ~np.isfinite(err) | (err <= 0) | ~on_chip  # gaps: no weight
    return np.where(bad, 0.0, flux), np.where(bad, np.inf, err)


def cheb_normalise(flux, err, cont, order=2):
    """Divide by a Chebyshev polynomial of the given order fitted, per detector, to the continuum pixels `cont`
    (boolean, NPIX) with weights 1/err^2 (N15 sect. 2.3 and 5.3). Returns normalised flux and error;
    pixels outside the detectors get err = inf."""
    fn, en = np.zeros_like(flux), np.full_like(err, np.inf)
    for lo, hi in CHIPS:
        c = (wave > lo) & (wave < hi)
        t = (wave[c] - (lo + hi) / 2) / ((hi - lo) / 2)          # wavelength mapped to [-1, 1]
        B = np.polynomial.chebyshev.chebvander(t, order)         # (pixels, order+1) basis
        use = cont[c]
        w = 1 / err[:, c][:, use]**2                             # (N, continuum pixels); 0 for flagged pixels
        # one small weighted least-squares problem per star: (B^T W B) a = B^T W f
        BtWB = np.einsum('pi,np,pj->nij', B[use], w, B[use])
        BtWf = np.einsum('pi,np->ni', B[use], w * flux[:, c][:, use])
        a = np.linalg.solve(BtWB, BtWf[:, :, None])[:, :, 0]     # (N, order+1); b as a stack of column vectors
        cont_fit = a @ B.T                                       # (N, pixels of this detector)
        fn[:, c] = flux[:, c] / cont_fit
        en[:, c] = err[:, c] / cont_fit
    return fn, en
# --- The Cannon (as written in Chapter 2) -----------------------------------------------------------------
from scipy.optimize import least_squares


def vectorize(x):
    """Quadratic label vector v(x) = [1, x_k, x_k x_l (k <= l)] for scaled labels x of shape (N, K): (N, D)."""
    x = np.atleast_2d(x)
    k, l = np.triu_indices(x.shape[1])
    return np.hstack([np.ones((len(x), 1)), x, x[:, k] * x[:, l]])


def dvectorize(x):
    """Derivative dv/dx of the label vector of one star (x of shape (K,)): (D, K)."""
    K = len(x)
    k, l = np.triu_indices(K)
    J = np.zeros((1 + K + len(k), K))
    J[1:1 + K] = np.eye(K)
    for i, (a, b) in enumerate(zip(k, l)):
        J[1 + K + i, a] += x[b]
        J[1 + K + i, b] += x[a]
    return J


def fit_theta(V, flux, var):
    """Weighted least squares at every pixel: theta (P, D) and its covariance (P, D, D); weights 1/var (N, P).
    Zero-weight entries (var = inf) drop out of the sums."""
    w = 1 / var
    flux = np.where(w > 0, flux, 0.0)                       # NaN flux at zero weight must not poison the sums
    N, Dv = V.shape
    A = ((V[:, :, None] * V[:, None, :]).reshape(N, Dv * Dv).T @ w).T.reshape(-1, Dv, Dv)
    b = (V.T @ (w * flux)).T
    cov = np.linalg.pinv(A)                                 # pinv: pixels without data give theta = 0, not an error
    return np.einsum('pde,pe->pd', cov, b), cov


S_GRID = np.r_[0, np.logspace(-4, 0, 41)]                  # 0, then 1e-4 ... 1 in steps of 0.1 dex


def fit_scatter(resid, err2):
    """Maximum-likelihood scatter s per pixel for fixed residuals: best value on the grid S_GRID, refined by a
    parabola in log s (Chapter 2). Zero-weight entries (err2 = inf) are ignored."""
    use = np.isfinite(err2) & np.isfinite(resid)
    r2 = np.where(use, resid**2, 0.0)
    e2 = np.where(use, err2, 1.0)
    nll = np.array([np.sum(use * (r2 / (e2 + s**2) + np.log(e2 + s**2)), axis=0) for s in S_GRID])
    best = np.argmin(nll, axis=0)
    j = np.clip(best, 2, len(S_GRID) - 2)
    p = np.arange(nll.shape[1])
    a, b, c = nll[j - 1, p], nll[j, p], nll[j + 1, p]
    shift = np.clip(0.5 * (a - c) / np.where(a - 2 * b + c > 0, a - 2 * b + c, np.inf), -1, 1)   # stay near the grid optimum
    return np.where(best == 0, 0.0, S_GRID[j] * np.exp(shift * np.log(S_GRID[2] / S_GRID[1])))


def train(x, flux, err, n_iter=3):
    """Training step. x: scaled labels (N, K); flux, err: (N, P) (err = inf for pixels to ignore).
    Returns theta (P, D), its covariance (P, D, D) and the scatter s (P,)."""
    V, err2 = vectorize(x), err**2
    s = np.zeros(flux.shape[1])
    for _ in range(n_iter):
        theta, cov = fit_theta(V, flux, err2 + s**2)
        s = fit_scatter(flux - V @ theta.T, err2)
    theta, cov = fit_theta(V, flux, err2 + s**2)
    return theta, cov, s


def test(flux, err, theta, s, x0=None):
    """Test step for one spectrum: scaled labels (K,), covariance (K, K), chi^2, number of pixels used, converged?"""
    K = int(round((np.sqrt(8 * theta.shape[1] + 1) - 3) / 2))   # invert D = 1 + K + K(K+1)/2
    ok = np.isfinite(flux) & np.isfinite(err) & (theta[:, 0] != 0)
    w = 1 / np.sqrt(err[ok]**2 + s[ok]**2)
    th, f = theta[ok], flux[ok]
    res = lambda x: (f - th @ vectorize(x)[0]) * w
    jac = lambda x: -(th @ dvectorize(x)) * w[:, None]
    r = least_squares(res, np.zeros(K) if x0 is None else x0, jac=jac, method='lm')
    # covariance from the Jacobian at the optimum (pinv: no crash if a label is unconstrained); r.success: converged
    return r.x, np.linalg.pinv(r.jac.T @ r.jac), np.sum(r.fun**2), ok.sum(), r.success


def train_clean(x, flux, err, cut=3.0):
    """Train, drop the training stars whose mean squared residual per pixel (in units of sigma^2 + s^2) exceeds
    `cut`, and train again (Chapter 4). Returns theta, its covariance, s, and the boolean mask of stars kept."""
    theta, cov, s = train(x, flux, err)
    z2 = (flux - vectorize(x) @ theta.T)**2 / (err**2 + s**2)
    z2 = np.where(np.isfinite(err) & (theta[:, 0] != 0), z2, np.nan)
    keep = np.nanmean(z2, axis=1) < cut
    theta, cov, s = train(x[keep], flux[keep], err[keep])
    return theta, cov, s, keep


def scatter_stats(d):
    """Summary of differences d (e.g. Cannon - reference): mean (bias), standard deviation, root-mean-square error,
    robust standard deviation 1.4826 x median absolute deviation (= std for a Gaussian, insensitive to outliers),
    and the fraction of points more than 3 robust std from the median (the tails)."""
    d = np.asarray(d)[np.isfinite(d)]
    med = np.median(d)
    rob = 1.4826 * np.median(np.abs(d - med))
    return {'bias': d.mean(), 'std': d.std(), 'rmse': np.sqrt(np.mean(d**2)), 'robust': rob,
            'tail': np.mean(np.abs(d - med) > 3 * rob)}
# --- More of The Cannon (Chapter 10): any polynomial order, censoring, L1 regularisation, several starting points,
# --- and a test of whether labels lie inside the training set -------------------------------------------------
from itertools import combinations_with_replacement
import warnings
from scipy.optimize import linprog


def label_terms(K, order=2):
    """The terms of the label vector after the leading 1: tuples of label indices, e.g. (0,), (0, 1), (2, 2, 2)."""
    return [t for p in range(1, order + 1) for t in combinations_with_replacement(range(K), p)]


def vectorize_n(x, order=2):
    """Label vector of polynomial order `order` for scaled labels x (N, K): (N, D). For order 2 it equals vectorize."""
    x = np.atleast_2d(x)
    return np.hstack([np.ones((len(x), 1))] + [np.prod(x[:, list(t)], axis=1, keepdims=True)
                                                for t in label_terms(x.shape[1], order)])


def dvectorize_n(x, order=2):
    """Derivative dv/dx of the label vector of one star (x of shape (K,)): (D, K)."""
    T = label_terms(len(x), order)
    J = np.zeros((1 + len(T), len(x)))
    for i, t in enumerate(T):
        for j in range(len(t)):                              # d/dx of x_a x_b x_c ...: drop one factor at a time
            J[1 + i, t[j]] += np.prod(x[list(t[:j] + t[j + 1:])])
    return J


def censor_mask(allowed, order=2):
    """Coefficients allowed at each pixel, (P, D), from the labels allowed at each pixel, (P, K): a term may be
    non-zero only if all the labels in it are allowed. The leading coefficient (the mean spectrum) always is."""
    T = label_terms(allowed.shape[1], order)
    return np.hstack([np.ones((len(allowed), 1), bool)] + [allowed[:, list(t)].all(axis=1, keepdims=True) for t in T])


def fit_theta_l1(V, flux, var, lam=0.0, mask=None, theta0=None, chunk=2048, tol=1e-8, max_iter=20000, kappa=0.1):
    """At every pixel minimise chi^2 + lam * sum_{d >= 1} |theta_d| (Casey et al. 2016), with theta_d = 0 where
    mask (P, D) is False. lam = 0: weighted least squares, solved directly. lam > 0: ADMM (Boyd et al. 2011, §6.4) on rescaled coefficients,
    all pixels of a chunk at once, started from theta0 (P, D) if given. Weights 1/var (N, P); zero-weight entries drop
    out. Returns theta (P, D) and the number of ADMM iterations used (0 for lam = 0)."""
    N, D = V.shape
    P = flux.shape[1]
    mask = np.ones((P, D), bool) if mask is None else mask
    w = 1 / var
    flux = np.where(w > 0, flux, 0.0)
    VV = (V[:, :, None] * V[:, None, :]).reshape(N, D * D).T
    pen = np.r_[0.0, np.ones(D - 1)]                             # theta_0 (the mean spectrum) is not penalised
    theta, n_used = np.zeros((P, D)), 0
    for p0 in range(0, P, chunk):
        sl = slice(p0, p0 + chunk)
        A = (VV @ w[:, sl]).T.reshape(-1, D, D)                  # V^T W V at each pixel
        b = (V.T @ (w[:, sl] * flux[:, sl])).T                   # V^T W f
        m = mask[sl]
        if lam == 0:                                             # censored rows/columns replaced by the identity
            Am = A * (m[:, :, None] & m[:, None, :]) + np.eye(D) * ~m[:, None, :]
            theta[sl] = np.einsum('pde,pe->pd', np.linalg.pinv(Am), b * m) * m
            continue
        # Rescale each coefficient so that the curvature of chi^2 along it is 1: theta = d * y. Without this, linear and
        # quadratic terms have curvatures that differ by orders of magnitude and ADMM crawls. |theta_k| = d_k |y_k|.
        dg = np.einsum('pdd->pd', A)
        d = 1 / np.sqrt(np.where(dg > 0, 2 * dg, 1.0))
        As, bs = A * d[:, :, None] * d[:, None, :], b * d
        # ADMM: split y (quadratic part) from z (L1 part and mask), y = z; u is the scaled dual variable. The step
        # size rho scales with the soft threshold lam * d, which in these units sets how far the solution moves.
        rho = np.minimum(kappa * lam * np.median(d, axis=1), 1.0)[:, None]   # capped at the (unit) curvature
        M = np.linalg.inv(2 * As + rho[:, :, None] * np.eye(D))   # factorised once, used at every iteration
        z = np.zeros_like(b) if theta0 is None else theta0[sl] * m / d
        u = np.zeros_like(b)
        thr = pen * lam * d / rho                                # soft threshold for each scaled coefficient
        act = np.arange(len(b))                                  # pixels not yet converged
        for it in range(max_iter):
            y = np.einsum('pde,pe->pd', M[act], 2 * bs[act] + rho[act] * (z[act] - u[act]))
            v = y + u[act]
            z_new = np.where(m[act], np.sign(v) * np.maximum(np.abs(v) - thr[act], 0), 0.0)
            da = d[act]                                          # convergence judged on theta, not on y
            done = (np.abs(da * (y - z_new)).max(axis=1) < tol) & (np.abs(da * (z_new - z[act])).max(axis=1) < tol)
            z[act], u[act] = z_new, v - z_new
            act = act[~done]
            if len(act) == 0:
                break
        if len(act):
            warnings.warn(f'fit_theta_l1: {len(act)} pixels did not converge in {max_iter} iterations')
        n_used = max(n_used, it + 1)
        theta[sl] = z * d                                        # z has the exact zeros
    return theta, n_used


def median_scatter(resid, err2, n_bisect=50):
    """AnniesLasso's scatter: at each pixel the s >= 0 for which the median of resid^2 / (err2 + s^2) over the stars is
    1 (s = 0 where it is already below 1). Bisection on all pixels at once; zero-weight entries are ignored."""
    z = lambda s: np.nanmedian(np.where(np.isfinite(err2), resid**2 / (err2 + s**2), np.nan), axis=0)
    lo, hi = np.zeros(resid.shape[1]), np.full(resid.shape[1], 1.0)
    for _ in range(n_bisect):                                # median chi^2 decreases with s
        mid = (lo + hi) / 2
        big = z(mid) > 1
        lo, hi = np.where(big, mid, lo), np.where(big, hi, mid)
    return np.where(z(0.0) > 1, hi, 0.0)


def train_more(x, flux, err, lam=0.0, mask=None, order=2, n_iter=3, scatter='ml'):
    """Training step with polynomial order, censoring mask and L1 regularisation. scatter='ml': coefficients and
    maximum-likelihood scatter iterated, as in `train` (Chapter 2). scatter='median': AnniesLasso's recipe —
    coefficients fitted once with s = 0, then s from `median_scatter`. Returns theta (P, D) and s (P,)."""
    V, err2 = vectorize_n(x, order), err**2
    if scatter == 'median':
        theta = fit_theta_l1(V, flux, err2, lam, mask)[0]
        return theta, median_scatter(flux - V @ theta.T, err2)
    s, theta = np.zeros(flux.shape[1]), None
    for _ in range(n_iter):
        theta = fit_theta_l1(V, flux, err2 + s**2, lam, mask, theta)[0]     # each fit starts from the previous one
        s = fit_scatter(flux - V @ theta.T, err2)
    return fit_theta_l1(V, flux, err2 + s**2, lam, mask, theta)[0], s


def test_more(flux, err, theta, s, order=2, starts=None):
    """Test step for one spectrum, as `test` (Chapter 2) but for any polynomial order and from several starting
    points (rows of `starts`, scaled labels; default: the median labels, 0); of the fits that converged, the one with
    the lowest chi^2 is kept.
    Returns scaled labels, covariance, chi^2, number of pixels used, converged?"""
    K = next(k for k in range(1, 100) if 1 + len(label_terms(k, order)) == theta.shape[1])
    ok = np.isfinite(flux) & np.isfinite(err) & (theta[:, 0] != 0)
    w = 1 / np.sqrt(err[ok]**2 + s[ok]**2)
    th, f = theta[ok], flux[ok]
    res = lambda x: (f - th @ vectorize_n(x, order)[0]) * w
    jac = lambda x: -(th @ dvectorize_n(x, order)) * w[:, None]
    fits = [least_squares(res, x0, jac=jac, method='lm') for x0 in (np.zeros((1, K)) if starts is None else
                                                                     np.atleast_2d(starts))]
    r = min(fits, key=lambda r: (not r.success, np.sum(r.fun**2)))   # converged fits first, then lowest chi^2
    return r.x, np.linalg.pinv(r.jac.T @ r.jac), np.sum(r.fun**2), ok.sum(), r.success


def in_hull(x, X):
    """Is each row of x (M, K) inside the convex hull of the training labels X (N, K)? True if x is a weighted mean
    of training points with non-negative weights summing to 1: a linear-programming feasibility problem, which works
    in any number of dimensions (a Delaunay triangulation does not, beyond a few). Labels are scaled by the standard
    deviation of X so that kelvin and dex are treated alike."""
    sd = X.std(axis=0)
    A = np.vstack([(X / sd).T, np.ones(len(X))])
    status = np.array([linprog(np.zeros(len(X)), A_eq=A, b_eq=np.r_[p / sd, 1], bounds=(0, None), method='highs').status
                       for p in np.atleast_2d(x)])
    if not np.all((status == 0) | (status == 2)):              # 0: feasible (inside), 2: infeasible (outside)
        raise RuntimeError(f'linprog failed with status {set(status) - {0, 2}}')
    return status == 0
# --- Lux (Horta et al. 2025): labels and spectra both generated from a latent vector per star (Chapter 11) ----
# labels l_n = A z_n + noise (errors sl_n), fluxes f_n = B z_n + noise (errors s_n and pixel scatter s); every step of
# the training is a weighted linear least-squares problem, solved here in closed form.


def lux_weights(values, errors):
    """Inverse variances, 0 where the error is not finite and positive or the value is not finite (masked pixels,
    missing labels), and the values with those entries set to 0 so that they cannot produce NaN."""
    ok = np.isfinite(errors) & (errors > 0) & np.isfinite(values)
    w = np.where(ok, 1 / np.where(ok, errors, 1.0)**2, 0.0)
    return w, np.where(w > 0, values, 0.0)


def lsq_update(G, h, c0, incremental=True, rcond=1e-10):
    """Solve the normal equations G c = h for a stack of problems (G: (K, P, P), h and c0: (K, P)).
    incremental=True: c0 + G^+ (h - G c0), which has the same residual as the minimum-norm solution but keeps the part
    of c0 that the data do not constrain (as a Gauss-Newton step from c0 does); incremental=False: G^+ h.
    Directions with eigenvalues of G below rcond times the largest count as unconstrained (rcond = 0: none do, for G
    that is positive definite, e.g. with a prior)."""
    vals, vecs = np.linalg.eigh(G)
    inv = np.where(vals > rcond * np.max(vals, axis=1, keepdims=True), 1 / np.where(vals > 0, vals, 1.0), 0.0)
    r = h - np.einsum('kpq,kq->kp', G, c0) if incremental else h
    step = np.einsum('kpq,kq,kq->kp', vecs, inv, np.einsum('kqp,kq->kp', vecs, r))
    return c0 + step if incremental else step


def lux_a_step(Z, lab, wl, A, incremental=True):
    """Label coefficients A (M, P) at fixed latents Z (N, P): one weighted least-squares problem per label."""
    G = np.einsum('nm,np,nq->mpq', wl, Z, Z)
    return lsq_update(G, np.einsum('nm,np->mp', wl * lab, Z), A, incremental)


def lux_b_step(Z, flux, w, B, incremental=True):
    """Flux coefficients B (Lambda, P) at fixed latents Z: one weighted least-squares problem per pixel."""
    N, P = Z.shape
    G = ((Z[:, :, None] * Z[:, None, :]).reshape(N, P * P).T @ w).T.reshape(-1, P, P)
    return lsq_update(G, (Z.T @ (w * flux)).T, B, incremental)


def lux_z_step(A, B, lab, wl, flux, w, Z, tau=np.inf, incremental=True):
    """Latents Z (N, P) at fixed A, B: one weighted least-squares problem per star, using its labels (weights wl) and
    fluxes (weights w), with a Gaussian prior of width tau on every latent (ridge term)."""
    P = A.shape[1]
    G = (w @ (B[:, :, None] * B[:, None, :]).reshape(-1, P * P)).reshape(-1, P, P)
    G += np.einsum('nm,mp,mq->npq', wl, A, A) + np.eye(P) / tau**2
    rcond = 0.0 if np.isfinite(tau) else 1e-10                # with a prior every direction is constrained by it
    return lsq_update(G, (w * flux) @ B + (wl * lab) @ A, Z, incremental, rcond)


def lux_objective(A, B, Z, s, lab, wl, flux, w, tau=np.inf):
    """Q = -2 ln L (labels and fluxes, up to a constant) + sum |z|^2 / tau^2; masked entries (weight 0) excluded."""
    var = np.where(w > 0, 1 / np.where(w > 0, w, 1.0) + s**2, 1.0)
    q_flux = np.where(w > 0, (flux - Z @ B.T)**2 / var + np.log(var), 0.0)
    return np.sum(wl * (lab - Z @ A.T)**2) + np.sum(q_flux) + np.sum(Z**2) / tau**2


def lux_scatter_step(B, Z, flux, w, s):
    """Pixel scatter by maximum likelihood at fixed B, Z (`fit_scatter` of Chapter 2), kept at each pixel only if it
    does not increase that pixel's part of Q."""
    err2 = np.where(w > 0, 1 / np.where(w > 0, w, 1.0), np.inf)
    resid = flux - Z @ B.T
    q = lambda s_: np.sum(np.where(w > 0, resid**2 / (err2 + s_**2) + np.log(np.where(w > 0, err2, 1.0) + s_**2), 0.0),
                          axis=0)
    s_new = fit_scatter(resid, err2)
    return np.where(q(s_new) <= q(s), s_new, s)


def lux_init(lab, P, n_flux, seed=0):
    """H25 Table 1, eq. 9: z_n = [1, (l_n - median) / ((p97.5 - p2.5) / 4), 0, ...], A and B uniform in [0, 1].
    Pivot and scale come from the labels that are present (NaN = missing); missing entries start at 0."""
    rng = np.random.default_rng(seed)
    q = np.nanpercentile(lab, [2.5, 50, 97.5], axis=0)
    x = np.nan_to_num((lab - q[1]) / ((q[2] - q[0]) / 4))
    Z = np.hstack([np.ones((len(lab), 1)), x, np.zeros((len(lab), P - lab.shape[1] - 1))])
    return rng.uniform(0, 1, (lab.shape[1], P)), rng.uniform(0, 1, (n_flux, P)), Z


def lux_train(lab, lab_err, flux, err, P=49, n_iter=5, tau=1000.0, incremental=True, seed=0):
    """Training as described in H25 §4.2. Stage 1: n_iter rounds of A-, B- and z-steps without scatter or prior.
    Stage 2 (once): scatter, then B with the new scatter, then z with the new B, scatter and the prior.
    Labels missing as NaN or with infinite error are ignored. Returns A, B, s, Z and Q after every step."""
    wl, l0 = lux_weights(lab, lab_err)
    w, f0 = lux_weights(flux, err)
    A, B, Z = lux_init(np.where(wl > 0, lab, np.nan), P, flux.shape[1], seed)
    s, history = np.zeros(flux.shape[1]), []
    for _ in range(n_iter):
        A = lux_a_step(Z, l0, wl, A, incremental)
        B = lux_b_step(Z, f0, w, B, incremental)
        Z = lux_z_step(A, B, l0, wl, f0, w, Z, np.inf, incremental)
        history.append(lux_objective(A, B, Z, s, l0, wl, f0, w))
    s = lux_scatter_step(B, Z, f0, w, s)
    wv = w / (1 + w * s**2)                                   # 1 / (sigma^2 + s^2), 0 where masked
    B = lux_b_step(Z, f0, wv, B, incremental)
    Z = lux_z_step(A, B, l0, wl, f0, wv, Z, tau, incremental)
    history.append(lux_objective(A, B, Z, s, l0, wl, f0, w, tau))
    return A, B, s, Z, np.array(history)


def lux_test(flux, err, A, B, s, rcond=1e-12):
    """Test step (H25 eq. 10): latents of each star from its spectrum alone (no prior), labels = A z.
    Latent directions the spectrum does not constrain (eigenvalues of B^T W B below rcond times the largest) are left at
    zero; the labels do not depend on them only if A has no component along them, which is checked and returned.
    Returns labels (N, M), their formal covariance (N, M, M), and for each star the condition number of the retained
    part of B^T W B and the largest |A v| over the discarded directions v (0 if none)."""
    w, f0 = lux_weights(flux, err)
    wv = w / (1 + w * s**2)
    P = B.shape[1]
    G = (wv @ (B[:, :, None] * B[:, None, :]).reshape(-1, P * P)).reshape(-1, P, P)
    vals, vecs = np.linalg.eigh(G)
    keep = vals > rcond * vals[:, -1:]
    Ginv = np.einsum('npq,nq,nrq->npr', vecs, np.where(keep, 1 / np.where(keep, vals, 1.0), 0.0), vecs)
    z = np.einsum('npq,nq->np', Ginv, (wv * f0) @ B)
    cond = vals[:, -1] / np.min(np.where(keep, vals, np.inf), axis=1)
    leak = np.max(np.where(~keep[:, None, :], np.abs(np.einsum('mp,npq->nmq', A, vecs)), 0.0), axis=(1, 2))
    return z @ A.T, np.einsum('mp,npq,lq->nml', A, Ginv, A), cond, leak
import time

labels = load('labels')
lines = load('linelist')
cont = load('continuum_pixels')['continuum'].to_numpy()
EL = ['c', 'n', 'o', 'mg', 'al', 'si', 'ca', 'mn', 'ni']
NAMES = ['teff', 'logg', 'fe_h'] + [e + '_fe' for e in EL]
TEX = [r'$T_{\rm eff}$', r'$\log g$', '[Fe/H]'] + [f'[{e.capitalize()}/Fe]' for e in EL]
data = {}
for nm in ['train', 'test']:
    lab = labels[labels['sample'] == nm].reset_index(drop=True)
    f, e = cheb_normalise(*prepare(load(f'spectra_{nm}')), cont)
    good = np.isfinite(lab[NAMES]).all(axis=1).to_numpy()
    data[nm] = (lab[NAMES].to_numpy()[good], lab[[n + '_err' for n in NAMES]].to_numpy()[good], f[good], e[good])
L, Lerr, flux, err = data['train']
Lt, _, ft, et = (a[::2] for a in data['test'])               # every second test star (S/N 50-100), as in Chapter 10
val = np.random.default_rng(10).permutation(len(L)) % 5 == 0  # the validation stars of Chapter 10
tr = np.flatnonzero(~val)
print(f'training {len(tr)}, validation {val.sum()}, test {len(Lt)} stars; median label errors:',
      dict(zip(NAMES, np.median(Lerr, axis=0).round(3))))
rob = lambda d: np.array([scatter_stats(d[:, k])['robust'] for k in range(d.shape[1])])   # robust std per label
RES = {}                                                       # robust std on validation and test, per model


def score(name, lab_val, lab_test):
    """Robust std of each label minus ASPCAP, on validation and test stars; printed and stored."""
    RES[name] = {'validation': rob(lab_val - L[val]), 'test': rob(lab_test - Lt)}
    v, t = RES[name]['validation'], RES[name]['test']
    print(f'{name:38s} validation: T_eff {v[0]:5.1f} K, log g {v[1]:.3f}, [Fe/H] {v[2]:.3f}, median [X/Fe] '
          f'{np.median(v[3:]):.4f} | test: {t[0]:5.1f} K, {t[1]:.3f}, {t[2]:.3f}, {np.median(t[3:]):.4f}')


def cannon(idx, labs, error=err, error_t=et):
    """Twelve-label quadratic Cannon (Chapter 10) trained on stars idx with labels labs; labels of validation and test
    stars. error, error_t: flux errors of the training set and of the test stars (infinite where pixels are hidden)."""
    p, c = np.median(labs, axis=0), (np.percentile(labs, 97.5, axis=0) - np.percentile(labs, 2.5, axis=0)) / 2
    th, s = train_more((labs - p) / c, flux[idx], error[idx])
    run = lambda F, E: np.array([test_more(F[n], E[n], th, s)[0] for n in range(len(F))]) * c + p
    return run(flux[val], error[val]), run(ft, error_t)


def lux(idx, labs, labs_err, P=49, error=err, error_t=et, incremental=True):
    """Lux trained on stars idx; labels of validation and test stars, and the latents of the training stars."""
    A, B, s, Z, _ = lux_train(labs, labs_err, flux[idx], error[idx], P=P, incremental=incremental)
    out = [lux_test(F, E, A, B, s) for F, E in [(flux[val], error[val]), (ft, error_t)]]
    assert all(np.all(o[3] == 0) for o in out), 'some labels depend on latent directions the spectra do not constrain'
    return out[0][0], out[1][0], Z

## 11.1 Lux against The Cannon — and against ridge regression

Train Lux ($P = 49$, the value of H25's notebook; their paper says $4M = 48$) and the twelve-label Cannon of
Chapter 10 on the same 1308 stars, and label the validation and test stars. Lux's labels are a *linear* function of
the flux, $\boldsymbol\ell = A(B^\top W B)^{-1}B^\top W\mathbf f$. The simplest model with that property is ridge
regression of the labels on the fluxes (Hoerl & Kennard 1970): it is the baseline any linear method must beat. Its
one parameter is chosen on the validation stars, so ridge's validation scores are slightly optimistic; its test
scores are not. (As in Chapter 10, the validation stars helped to choose the continuum pixels in Chapter 3, which
Chapter 5 found to change the precision by less than 2%.)

In [ ]:
t0 = time.time()
score('Cannon (12 labels, quadratic)', *cannon(tr, L[tr]))
lv, lt, Z49 = lux(tr, L[tr], Lerr[tr])
score('Lux (P = 49)', lv, lt)
fill = lambda f, e: np.where(np.isfinite(e), f, 1.0)            # ridge needs a complete flux matrix


def ridge(idx, labs, lam, E=err, E_t=et):
    """Ridge regression of labels on fluxes (dual form: N x N system); labels of validation and test stars."""
    X = fill(flux[idx], E[idx])
    mu, ym = X.mean(axis=0), labs.mean(axis=0)
    W = (X - mu).T @ np.linalg.solve((X - mu) @ (X - mu).T + lam * np.eye(len(X)), labs - ym)
    return (fill(flux[val], E[val]) - mu) @ W + ym, (fill(ft, E_t) - mu) @ W + ym


spread = np.percentile(L[tr], 97.5, axis=0) - np.percentile(L[tr], 2.5, axis=0)
lam_r = min([1e-2, 1e-1, 1, 10], key=lambda lam: np.median(rob(ridge(tr, L[tr], lam)[0] - L[val]) / spread))
score(f'ridge regression (lambda = {lam_r:g})', *ridge(tr, L[tr], lam_r))
print(f'{time.time() - t0:.0f} s')

**How many latent dimensions does Lux use?** The starting latents $[1, \text{scaled labels}, 0, \dots, 0]$ have only
$M + 1 = 13$ non-zero columns. Fitting $A$ and $B$ to them is a least-squares problem with 49 unknowns per label or
pixel of which only 13 are constrained. The usual "minimum-norm" solution sets the other 36 to zero, and the next
latent fit then keeps the extra 36 latent coordinates at zero too: the model stays at 13 dimensions whatever $P$ is.
H25's optimiser instead updates from the current (random) values, keeping their unconstrained part, which activates
the extra dimensions; our `lsq_update(..., incremental=True)` does the same. Compare.

In [ ]:
t0 = time.time()
ranks = {}
for P, inc in [(13, True), (25, True), (49, False)]:
    name = f'Lux (P = {P}' + (', minimum-norm steps)' if not inc else ')')
    lv_, lt_, Z = lux(tr, L[tr], Lerr[tr], P=P, incremental=inc)
    score(name, lv_, lt_)
    ranks[name] = np.linalg.matrix_rank(Z)
ranks['Lux (P = 49)'] = np.linalg.matrix_rank(Z49)
print('rank of the training latents:', ranks)
print(f'{time.time() - t0:.0f} s')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
models = ['Cannon (12 labels, quadratic)', f'ridge regression (lambda = {lam_r:g})', 'Lux (P = 13)', 'Lux (P = 25)',
          'Lux (P = 49)', 'Lux (P = 49, minimum-norm steps)']
short = ['Cannon', 'ridge', 'Lux P=13', 'Lux P=25', 'Lux P=49', 'Lux P=49\nmin-norm']
for a, (k, lab) in zip(ax, [(0, r'$T_{\rm eff}$ [K]'), (1, r'$\log g$'), (None, 'median of the nine [X/Fe]')]):
    for j, split in enumerate(['validation', 'test']):
        v = [RES[m][split][k] if k is not None else np.median(RES[m][split][3:]) for m in models]
        a.bar(np.arange(len(models)) + 0.38 * j - 0.19, v, 0.36, color=['C0', 'C1'][j],
              label=split + (' (high S/N)' if j == 0 else ' (S/N 50–100)'))
    a.set_xticks(range(len(models)), short, fontsize=8)
    a.set(ylabel=f'robust std vs ASPCAP, {lab}')
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Figure 11.1.** Precision of $T_{\rm eff}$, $\log g$ and the abundances (median over the nine [X/Fe]) for the
twelve-label Cannon, ridge regression of the labels on the fluxes, and Lux with $P = 13, 25, 49$ latent dimensions,
all trained on the same 1308 stars; blue: validation stars (high S/N), orange: test stars (S/N 50–100). Lux and
ridge — both linear in the flux — measure the abundances better than The Cannon and the stellar parameters worse,
much worse at low S/N, where a linear model's many coefficients also fit noise. Most of Lux's abundance gain is
already in ridge regression. Lux with $P = 49$ and minimum-norm steps gives *exactly* the same numbers as $P = 13$:
its latents have rank 13 (printed above).

**Checks.**

In [ ]:
cn, lx, rg = RES['Cannon (12 labels, quadratic)'], RES['Lux (P = 49)'], RES[f'ridge regression (lambda = {lam_r:g})']
assert ranks['Lux (P = 49, minimum-norm steps)'] == 13 and ranks['Lux (P = 49)'] == 49
assert np.allclose(RES['Lux (P = 49, minimum-norm steps)']['validation'], RES['Lux (P = 13)']['validation'])
assert np.median(lx['validation'][3:]) < np.median(cn['validation'][3:]), 'Lux abundances not better than Cannon'
assert lx['test'][0] > 1.3 * cn['test'][0], 'Lux T_eff at low S/N no longer clearly worse than Cannon'
assert np.median(rg['validation'][3:]) < np.median(cn['validation'][3:])

## 11.2 Measured or predicted?

Chapter 8 showed that [Mg/Fe] can be *predicted* from the other labels, because the abundances of disc stars are
correlated. Is Lux's better [Mg/Fe] read from the Mg lines? Hide every pixel within 1.5 Å of the 40 strongest Mg I
lines of our line list (weaker Mg lines and blends remain), in training and test spectra, and retrain all three
models.

In [ ]:
t0 = time.time()
w_mg = lines[lines.species == 'MG I'].wave_vac.to_numpy()
mg = np.min(np.abs(wave[:, None] - w_mg[None, :]), axis=1) < 1.5
err_nomg, et_nomg = np.where(mg[None, :], np.inf, err), np.where(mg[None, :], np.inf, et)
print(f'hiding {mg.sum()} pixels near Mg I lines')
score('Cannon, Mg lines hidden', *cannon(tr, L[tr], err_nomg, et_nomg))
score('Lux (P = 49), Mg lines hidden', *lux(tr, L[tr], Lerr[tr], error=err_nomg, error_t=et_nomg)[:2])
score('ridge, Mg lines hidden', *ridge(tr, L[tr], lam_r, err_nomg, et_nomg))
k = NAMES.index('mg_fe')
pairs = [('Cannon (12 labels, quadratic)', 'Cannon, Mg lines hidden'),
         (f'ridge regression (lambda = {lam_r:g})', 'ridge, Mg lines hidden'),
         ('Lux (P = 49)', 'Lux (P = 49), Mg lines hidden')]
cost = {(a_, sp): RES[b_][sp][k] / RES[a_][sp][k] - 1 for a_, b_ in pairs for sp in ['validation', 'test']}
print('increase of the [Mg/Fe] scatter when the Mg I windows are hidden:',
      {f'{a_.split()[0]}, {sp}': f'{v:+.0%}' for (a_, sp), v in cost.items()})
print(f'{time.time() - t0:.0f} s')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for i, (a_, b_) in enumerate(pairs):
    for j, split in enumerate(['validation', 'test']):
        x = i + 0.38 * j - 0.19
        ax.bar(x, RES[a_][split][k], 0.36, color=['C0', 'C1'][j], label=f'{split}, all pixels' if i == 0 else None)
        ax.plot([x - 0.17, x + 0.17], [RES[b_][split][k]] * 2, 'k-', lw=2,
                label='Mg I lines hidden' if i == 0 and j == 0 else None)
ax.set_xticks(range(3), ['Cannon', 'ridge', 'Lux'])
ax.set(ylabel='robust std of [Mg/Fe] vs ASPCAP [dex]')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Figure 11.2.** Precision of [Mg/Fe] for the three models with all pixels (bars) and with the 393 pixels around the
40 strongest Mg I lines hidden in training and test (black lines). All three read those lines — hiding them costs
20–43% (printed above) — but Lux with them hidden is still better than The Cannon with all pixels, on validation and
test stars. Much of the precision of [Mg/Fe], for every model, therefore comes from other pixels: weaker Mg lines and
blends that remain, and, as Chapter 8 found, the correlations of the training labels. "Better abundances" of a
data-driven model must be read with this in mind (Ness et al. 2019, 2022).

**Checks.**

In [ ]:
assert all(v > 0.1 for v in cost.values()), 'hiding the Mg I windows does not cost every model > 10%'
for sp in ['validation', 'test']:
    assert RES['Lux (P = 49), Mg lines hidden'][sp][k] < RES['Cannon (12 labels, quadratic)'][sp][k]

## 11.3 Missing labels

Because Lux treats labels as data with errors, a star with a missing label can still be used: its error is set to
infinity and the label drops out of the fit (the pivots, scales and starting latents use only the labels present).
The Cannon needs every label of every training star. Hide [N/Fe] for a random half of the training stars (each star
with probability ½).

In [ ]:
t0 = time.time()
miss = np.random.default_rng(20).random(len(tr)) < 0.5
L_miss = L[tr].copy()
L_miss[miss, NAMES.index('n_fe')] = np.nan
print(f'[N/Fe] hidden for {miss.sum()} of {len(tr)} training stars; {(~miss).sum()} keep it')
score('Lux, all stars, half without [N/Fe]', *lux(tr, L_miss, Lerr[tr])[:2])
score('Lux, only the stars with [N/Fe]', *lux(tr[~miss], L[tr][~miss], Lerr[tr][~miss])[:2])
score('Cannon, only the stars with [N/Fe]', *cannon(tr[~miss], L[tr][~miss]))
kn = NAMES.index('n_fe')
for m in ['Lux, all stars, half without [N/Fe]', 'Lux, only the stars with [N/Fe]', 'Cannon, only the stars with [N/Fe]']:
    print(f"{m:38s} [N/Fe]: validation {RES[m]['validation'][kn]:.3f}, test {RES[m]['test'][kn]:.3f}")
print(f'{time.time() - t0:.0f} s')

Training Lux on all 1308 stars, 682 of them without [N/Fe], gives the same precision as training it on the 626
stars with [N/Fe] only: in this random draw the extra spectra do not help. The Cannon trained on those 626 stars
measures [N/Fe] about as well. Being able to use incomplete labels is a convenience; here it brought no measurable
gain (one draw; other data, or a label missing for most stars, may differ).

**Check.**

In [ ]:
a_, b_ = RES['Lux, all stars, half without [N/Fe]'], RES['Lux, only the stars with [N/Fe]']
assert abs(a_['validation'][kn] / b_['validation'][kn] - 1) < 0.1

## 11.4 Noisy training labels

Real training labels have errors, and a cheaper survey's are larger than ASPCAP's (median 8 K in $T_{\rm eff}$,
0.01–0.02 dex in abundances). Add Gaussian noise of five times the ASPCAP error to the training labels — about 40 K,
0.12 dex in $\log g$ and 0.07 dex in abundances — and tell Lux the true total error, $\sqrt{26}\,\sigma$. Score all
models against the original ASPCAP labels of the validation and test stars.

In [ ]:
t0 = time.time()
noisy = L[tr] + np.random.default_rng(21).normal(0, 1, L[tr].shape) * 5 * Lerr[tr]
score('Cannon, noisy training labels', *cannon(tr, noisy))
score('ridge, noisy training labels', *ridge(tr, noisy, lam_r))
score('Lux, noisy training labels', *lux(tr, noisy, np.sqrt(26) * Lerr[tr])[:2])
print(f'{time.time() - t0:.0f} s')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
trio = [('Cannon (12 labels, quadratic)', 'Cannon, noisy training labels', 'Cannon'),
        (f'ridge regression (lambda = {lam_r:g})', 'ridge, noisy training labels', 'ridge'),
        ('Lux (P = 49)', 'Lux, noisy training labels', 'Lux')]
for a, (k_, lab) in zip(ax, [(None, 'median of the nine [X/Fe] [dex]'), (0, r'$T_{\rm eff}$ [K]')]):
    for j, split in enumerate(['validation', 'test']):
        for i, (c_, n_, nm) in enumerate(trio):
            get = lambda m: np.median(RES[m][split][3:]) if k_ is None else RES[m][split][k_]
            x = i + 0.38 * j - 0.19
            a.bar(x, get(c_), 0.36, color=['C0', 'C1'][j], alpha=0.5, label=f'{split}, ASPCAP labels' if i == 0 else None)
            a.plot([x - 0.17, x + 0.17], [get(n_)] * 2, 'k-', lw=2,
                   label='noisy training labels (5 sigma)' if i == 0 and j == 0 else None)
    a.set_xticks(range(3), [t[2] for t in trio])
    a.set(ylabel=f'robust std vs ASPCAP, {lab}')
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Figure 11.3.** Precision of the abundances (left) and of $T_{\rm eff}$ (right) when the models are trained on the
ASPCAP labels (bars) and on the same labels with noise of 5× the ASPCAP errors added (black lines). The Cannon's
abundances become twice as scattered. Its labels are *inputs* to the polynomial, and the degradation is what noise
in inputs does to fitted coefficients (the errors-in-variables problem; the regression dilution of Chapter 2),
although this experiment does not isolate that cause. Ridge regression, which treats the labels as outputs, suffers
less; Lux, which treats them as outputs with known errors and passes them through a low-dimensional latent space,
hardly at all — which of these features matters most, this comparison cannot say. This is the clearest advantage of
Lux on our data. Caveats: one noise draw; Lux was told the true total errors; ridge's strength was chosen on the
clean labels and not re-tuned. Appendix C tests each of these limitations explicitly.
With three new draws it confirms the large Cannon degradation and much smaller Lux changes, including when
Lux is told underestimated errors. Thus accurate label uncertainties alone do not explain the robustness.
Read the robustness claim here as a comparison with The Cannon; Appendix C supplies the retuned ridge control.

**Checks.**

In [ ]:
mx = lambda m, split: np.median(RES[m][split][3:])
for split in ['validation', 'test']:
    assert mx('Cannon, noisy training labels', split) > 1.5 * mx('Cannon (12 labels, quadratic)', split)
    assert mx('Lux, noisy training labels', split) < 1.15 * mx('Lux (P = 49)', split)
    assert mx('ridge, noisy training labels', split) > mx('Lux, noisy training labels', split)

## Summary

- Lux generates labels and spectra from one latent vector per star; trained by a sequence of weighted least-squares
  fits (and a grid fit of the scatter), it takes half a minute on 1308 stars. For the same least-squares steps our
  sixty lines agree with the *pollux* package to $10^{-9}$ (Appendix B).
- How an under-determined least-squares step is solved matters: minimum-norm steps leave the model at $M + 1 = 13$
  latent dimensions whatever $P$ is.
- On our data Lux measures abundances better than The Cannon and stellar parameters worse (much worse at low S/N);
  ridge regression gets most of the way, and much of the [Mg/Fe] precision survives hiding the strongest Mg lines.
- Training on stars with a missing label works, but did not help in our test.
- Lux's clear strength is robustness to noisy training labels: at 5× the ASPCAP errors its abundances barely change,
  while The Cannon's scatter doubles.
- For research use: the *pollux* package (Price-Whelan), which implements Lux and The Cannon in JAX with priors,
  probabilistic fits and any number of outputs — Appendix B shows how to use it.

## References

- Hoerl, A. E., & Kennard, R. W. 1970, *Ridge regression: biased estimation for nonorthogonal problems*,
  Technometrics 12, 55, [doi:10.1080/00401706.1970.10488634](https://doi.org/10.1080/00401706.1970.10488634).
- Horta, D., Price-Whelan, A. M., Hogg, D. W., Ness, M. K., & Casey, A. R. 2025, *Lux: a generative, multi-output,
  latent-variable model for astronomical data with noisy labels*,
  [AJ 169, 314](https://ui.adsabs.harvard.edu/abs/2025AJ....169..314H) — code:
  [github.com/DHortaD/Lux](https://github.com/DHortaD/Lux) (MIT licence), [ascl:2510.013](https://ui.adsabs.harvard.edu/abs/2025ascl.soft10013H).
- Ness, M., Hogg, D. W., Rix, H.-W., Ho, A. Y. Q., & Zasowski, G. 2015, *The Cannon: a data-driven approach to
  stellar label determination*, [ApJ 808, 16](https://ui.adsabs.harvard.edu/abs/2015ApJ...808...16N).
- Ness et al. 2019, *In the Galactic disk, stellar [Fe/H] and age predict orbits and precise [X/Fe]*,
  [ApJ 883, 177](https://ui.adsabs.harvard.edu/abs/2019ApJ...883..177N).
- Ness et al. 2022, *The homogeneity of the star-forming environment of the Milky Way disk over time*,
  [ApJ 926, 144](https://ui.adsabs.harvard.edu/abs/2022ApJ...926..144N).
- Price-Whelan, A. M., *pollux: a JAX framework for data-driven latent variable models*,
  [github.com/adrn/pollux](https://github.com/adrn/pollux), [documentation](https://pollux-astro.readthedocs.io).